### 실습 환경 준비

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv

load_dotenv("../.env")

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"prompt": "passage: ", "normalize_embeddings": True},
    query_encode_kwargs={"prompt": "query: ", "normalize_embeddings": True}
)

db = Chroma(
    persist_directory="../ch06_advanced_rag/chroma_db_e5",
    collection_name="langchain-docs",
    embedding_function=embeddings,
    create_collection_if_not_exists=False,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
model = ChatOpenAI(model="gpt-6-luna")
retriever = db.as_retriever()

prompt = ChatPromptTemplate.from_template('''
다음 문맥만을 고려해 질문에 답하세요.

문맥: """
{context}
"""

질문: {question}
''')

chain = {
    "question": RunnablePassthrough(),
    "context": retriever
} | prompt | model

### 피드백 버튼을 표시하는 함수 구현

In [11]:
from uuid import UUID
import ipywidgets as widgets
from IPython.display import display
from langsmith import Client

def display_feedback_buttons(run_id: UUID) -> None:
    # Good 버튼과 Bad 버튼 준비
    good_button = widgets.Button(
        description="Good",
        button_style="success",
        icon="thumbs-up"
    )
    bad_button = widgets.Button(
        description="Bad",
        button_style="danger",
        icon="thumbs-down"
    )

    # 클릭됐을 때 실행되는 함수 정의
    def on_button_clicked(button: widgets.Button) -> None:
        if button == good_button:
            score = 1
        elif button == bad_button:
            score = 0
        else:
            raise ValueError(f"Unknown button: {button}")

        langsmith_client = Client()
        langsmith_client.create_feedback(run_id=run_id, key="thumbs", score=score)
        print("피드백을 전송했습니다")

    # 버튼이 클릭됐을 때 on_button_clicked 함수 실행
    good_button.on_click(on_button_clicked)
    bad_button.on_click(on_button_clicked)

    # 버튼 표시
    display(good_button, bad_button)

### 피드백 버튼 표시

In [12]:
# 사용자 입력을 바탕으로 RAG Chain을 실행하고 피드백 버튼을 표시하는 코드를 작성
from langchain_core.tracers.context import collect_runs

# LangSmith의 트레이스 ID(Run ID)를 얻기 위해 collect_runs 함수 사용
with collect_runs() as runs_cb:
    output = chain.invoke("LangChain의 개요를 알려줘")
    print(output.content)
    run_id = runs_cb.traced_runs[0].id

display_feedback_buttons(run_id)

LangChain은 대규모 언어 모델(LLM) 기반 애플리케이션을 개발하기 위한 프레임워크입니다. 모델, 임베딩, 벡터 저장소 등 여러 구성 요소와 연동을 표준화하고 연결해 AI 애플리케이션 개발을 돕습니다. 다양한 데이터 소스를 활용하거나 모델을 쉽게 교체할 수 있으며, 폭넓은 통합 기능을 제공합니다.

복잡한 에이전트를 세밀하게 오케스트레이션하고 싶다면 LangGraph를 함께 사용할 수 있지만, LangGraph는 LangChain 없이도 사용할 수 있습니다.


Button(button_style='success', description='Good', icon='thumbs-up', style=ButtonStyle())

Button(button_style='danger', description='Bad', icon='thumbs-down', style=ButtonStyle())